# FX Pairs Trading — Model Training Pipeline V6 (TFT / NBEATSx / NHITS / PatchTST / Spacetimeformer → Leaderboard)

This notebook is organized as a **linear, step-by-step pipeline**. Run the cells top to bottom.
It assumes the training CSV (`dcc_garch_calander_gdelt_final.csv`) has already been produced by
your separate data-pipeline notebook.

All fixes from the review are baked in:
- NeuralForecast `val_size` corrected to match the declared val split
- Spacetimeformer trained on train+val only (never touches test), checkpointed on best val_loss
- Spacetimeformer gets a real, one-time test evaluation (previously missing entirely)
- NBEATSx / NHITS / PatchTST get an explicit, matched-to-real-targets test evaluation
- Quantile-column selection is exact (no fragile substring matching)
- A single unified leaderboard compares every model on the *same* held-out days


## V4 changelog — what changed vs V3

**All four models now train on the same three feature families where the architecture allows it:**
price/target history, calendar features, and GDELT news features (plus the DCC-GARCH `rho_`/`sigma_`
macro columns, tracked separately as `macro_cols`).

1. **Calendar features were never wired into *any* model in V3.** Step 2 now auto-detects
   `calendar_cols` by name pattern (day-of-week, month, quarter, holiday flags, etc.) — verify the
   printed list against your actual CSV columns before trusting it.
2. **Calendar features are *known in advance*, GDELT/macro features are not.** This is a real
   distinction, not a style choice: you can compute "next Friday is a holiday" today, but you can't
   compute tomorrow's GDELT shock score today. V3 dumped everything into "unknown" for TFT and
   `hist_exog_list` for NeuralForecast. V4 splits this correctly:
   - TFT: `calendar_cols` → `time_varying_known_reals`, `gdelt_cols + macro_cols` stay in
     `time_varying_unknown_reals`.
   - NeuralForecast: `calendar_cols` → `futr_exog_list`, `gdelt_cols + macro_cols` stay in
     `hist_exog_list`.
3. **NeuralForecast never received the `rho_`/`sigma_` macro columns at all in V3** (only TFT did) —
   both now get identical `gdelt_cols + macro_cols` exogenous inputs, so the TFT-vs-NF comparison is
   an apples-to-apples "same features, different architecture" comparison.
4. **Spacetimeformer still cannot be safely rewritten for real exogenous input** — see the large
   comment in Step 7 (Cell J). The class in this notebook (`stf.model.Spacetimeformer` with
   `d_x`/`d_y`/`max_seq_len`/`out_len`, single-tensor `forward`) does not match the published
   `spacetimeformer` library's actual `Spacetimeformer_Forecaster` API
   (`x_context, y_context, x_target, y_target`, separate constructor args). I could not confirm from
   the installed environment what this class actually accepts for `d_x != d_y`, so I did **not**
   guess-rewrite it. Instead:
   - It stays a documented **pure-autoregressive baseline** by default (`STF_USE_EXOG = False`) —
     target history only, same as V3, but now clearly labeled as intentionally starved rather than
     silently incomplete.
   - A best-effort exogenous path is written and gated behind `STF_USE_EXOG = True`, with a
     `try/except` fallback to the autoregressive path if the shape mismatch fails.
   - **Before flipping that flag**, run the diagnostic in the comment block and confirm
     `Spacetimeformer.__init__`/`forward` actually accept `d_x != d_y`. If you skip this, you'll get
     either a crash (safe) or, worse, a shape-broadcast that silently trains on garbage (not safe) —
     so don't skip it.
5. Leaderboard and manifest (Steps 11–12) now record which feature families each model actually
   received, so the comparison table is self-documenting instead of assuming equivalence.


## V5 changelog — real-data-driven fixes (vs V4)

V4's `macro_cols` detection was wrong against your actual CSV. Two real bugs, found from your
column dump and diagnostics, corrected here:

1. **V4 completely missed 277 `macro_*` columns.** V4's `macro_cols` only matched `rho_`/`sigma_`
   (DCC-GARCH). Your data also has a separate macroeconomic-indicator family
   (`macro_<ticker>=X`, `macro_surprise_<ticker>=X`, plus `_growth/_inflation/_labor/_pmi/_rate/_trade/_3d`
   sub-features per pair) that never reached any model. Split into two clearly-named variables now:
   `dcc_garch_leg_cols` (correlation/vol) and `macro_leg_cols` (macroeconomic indicators) — see #3.

2. **Leak guard for `_lead1` columns.** `macro_<ticker>=X_lead1` / `macro_surprise_<ticker>=X_lead1`
   are shifted-forward values — the same leak category as `event_lead_1/2`, already removed
   elsewhere in this pipeline. These are never included in any feature list.

3. **Per-group leg restriction (your call).** `group_id` is a 3-pair triplet
   (e.g. `AUDCAD=X_AUDUSD=X_NZDCAD=X`), but the raw `macro_*`/`rho_`/`sigma_` columns cover all 20
   underlying pairs. Feeding all 277+ columns to every group would hand each group's model data
   about pairs it has nothing to do with, and would badly blow up the feature/row ratio against
   `max_encoder_length=60`. Step 2 now parses each group's 3 legs from `group_id` and builds
   generic `leg1_*`/`leg2_*`/`leg3_*` columns (same column names across every group, but each row
   only sees its *own* group's legs) — 30 macro columns/group instead of 277, and 6 DCC-GARCH
   columns/group (3 pairwise `rho_leg*_leg*` + 3 `sigma_leg*`) instead of the full cross-pair set.
   **Verify the "missing source column" warnings Step 2 prints** — the `sigma_` naming convention
   was inferred (`sigma_<bare_ticker>`, no `=X`) from the `rho_` pattern in your diagnostics, not
   confirmed against an actual `sigma_` column name.

4. `event_lag_2` exists in your data (confirmed in your diagnostics) but was never used anywhere in
   V3/V4 — only `event_lag_1` was wired in. Added alongside it as a historical/unknown feature in
   TFT and NeuralForecast.

5. Your diagnostics show 44 dates where nearly all macro columns read exactly 0 (2013-03-22 through
   2024-01-17). That could be a genuine reading or a "no data yet" placeholder — **I did not
   auto-impute or drop these**, since I can't tell which from here. Worth checking before trusting
   early-period rows.


## V6 changelog — confirmed against the full 616-column dump (vs V5)

V5 flagged two things as unverified and left one scope decision open. All three are resolved here,
confirmed against your actual 616-column dump rather than inferred from a pattern.

1. **`sigma_` naming was wrong — now fixed.** V5 explicitly flagged this as an assumption
   ("`sigma_` is ASSUMED to follow the same bare-ticker convention... VERIFY this against the
   missing-source warning"). Your column dump confirms `sigma_` actually keeps the ticker's `=X`
   suffix (`sigma_AUDCAD=X`), unlike `rho_` which strips it (`rho_AUDCAD_AUDUSD`). V5's lookup used
   the bare-stripped ticker for both, which meant every `sigma_leg*` column silently resolved to
   nothing and was left NaN for the entire run — the DCC-GARCH volatility features never reached
   either model in V4 or V5, despite `dcc_garch_leg_cols` reporting a non-zero count. Step 2's
   `sigma_` lookup now uses the raw leg ticker (already carrying `=X`) instead of `_bare(ticker)`.

2. **`rho_*_change` / `rho_*_abs` were missing entirely.** The full column dump shows every `rho_`
   pair also has `_change` and `_abs` variants (e.g. `rho_AUDCAD_AUDUSD_change`,
   `rho_AUDCAD_AUDUSD_abs`) that V5's `dcc_garch_leg_cols` never looked for — same
   historical-derived treatment as the macro `_3d` variants already included. Added for all three
   leg pairs, taking `dcc_garch_leg_cols` from 6 columns/group to 15 (12 `rho_` variants + 3
   `sigma_leg*`).

3. **Four columns existed in the source CSV but reached neither model consistently (or, for
   `regime*`, reached only one) — now wired in, historical-only:**
   - `regime` / `regime_vol` / `regime_macro` — these were factorized and fed to NeuralForecast's
     `hist_exog_list` in V4/V5, but **never reached TFT at all**. Worse, the factorization ran as
     two independent `pd.factorize()` calls in two different cells — `pd.factorize()` gives no
     guarantee that the same category gets the same integer code across separate calls, so even if
     TFT *had* received these, there was no guarantee the two models were training on the same
     encoding of the same regime label. Fixed by factorizing once in Step 2
     (`regime_num`/`regime_vol_num`/`regime_macro_num`) and sharing that single encoding with both
     models.
   - `tone_shock_interaction`, `tone_mentions_interaction` — GDELT-derived interaction terms
     (tone × shock-count, tone × mentions-count) present in the CSV but missed by `gdelt_cols`'s
     `startswith()` pattern match. Folded directly into `gdelt_cols` rather than a separate list,
     since they're the same "group-level, not knowable in advance" shape as the rest of that family.
   - `macro_pressure` — a single **global** macro-pressure summary (same value for every group_id
     on a given date, unlike everything in `macro_leg_cols`, which is per-group). Kept in its own
     explicitly-named `global_macro_cols` rather than folded into `macro_leg_cols`, so that "one
     global column, not per-leg" stays visible at the call site.
   - `event_flag` — the V5 cell-C comment already described the intended treatment ("Both models
     now only see the lagged flag as a historical feature") but `event_flag` itself was never
     actually added to any feature list in V3–V5, only `event_lag_1`/`event_lag_2` were. Added as
     `extra_event_cols`, same historical-only treatment as the lag columns (never known-future).

   **Spacetimeformer (Step 7 / Cell J) deliberately does NOT receive these three additions.**
   `STF_USE_EXOG` still defaults to `False` pending confirmation of the installed library's real
   `forward()` signature (see Cell J's existing caveat, unchanged from V4/V5) — adding untested
   columns to an already-unconfirmed exogenous path would compound one open question with another.
   If you confirm the API and flip `STF_USE_EXOG = True`, decide then whether to fold
   `regime_cols`/`global_macro_cols`/`extra_event_cols` into `candidate_exog_cols` too (the folded-in
   `tone_shock_interaction`/`tone_mentions_interaction` are already covered, since they're inside
   `gdelt_cols`, which Spacetimeformer's exog path already considers).

4. Step 12's manifest (`feature_families`) now records `regime_cols`, `global_macro_cols`, and
   `extra_event_cols` alongside the existing families, so the saved run record stays
   self-documenting.


## Model Training & Evaluation

**Update:** removed a future-exogenous leak where `event_flag` (TFT: `event_lag_1` in `known_reals`; NeuralForecast: `event_flag` in `futr_exog_list`) was being treated as known ahead of time across the full forecast horizon -- it isn't, same reasoning as the data pipeline's earlier removal of `event_lead_1/2`. Both models now only see the lagged flag as a historical feature.\n\nAssumes `dcc_garch_calander_gdelt_final.csv` already exists (produced by your data-pipeline
notebook). **Make sure that notebook's `build_panel` merges `spread`/`zscore` into the panel** --
without it, Step 1 below will fail on the `assert "zscore" in df_raw.columns` check, since the
target is derived from `zscore`.

### Step 0 — Install dependencies

In [1]:
!pip install lightning neuralforecast pytorch_forecasting


### Step 0b — Mount Drive, global config, folder structure

In [9]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 0 — Mount + global config + folder structure for saved artifacts
# ══════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os

drive_base = "/content/drive/MyDrive/fx_models/"   # everything gets saved under here
DATA_DIR   = f"{drive_base}data/"
for sub in ["tft", "nbeats", "nhits", "patchtst", "spacetimeformer", "ensemble", "leaderboard"]:
    os.makedirs(f"{drive_base}{sub}/", exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

CSV_PATH = "/content/drive/MyDrive/dcc_garch_calander_gdelt_final.csv"

# ══════════════════════════════════════════════════════════════════════════
# Global random seed -- set ONCE, here, before any model touches randomness.
# Without this, rerunning the notebook shifts every model's numbers, which
# undermines any "model A vs model B" comparison: you can't tell whether a
# leaderboard change came from your edit or just from a different seed.
# ══════════════════════════════════════════════════════════════════════════
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    import lightning.pytorch as pl
    pl.seed_everything(SEED, workers=True)
except ImportError:
    pass

print(f"✅ Global random seed set to {SEED}")


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Global random seed set to 42


### Step 1 — Load data, build target, chronological per-group split, save splits

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# Cell A — Load, build target, chronological per-group split, SAVE splits
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import json
import pickle

df_raw = pd.read_csv(CSV_PATH)
df_raw["Date"] = pd.to_datetime(df_raw["Date"])
df_raw = df_raw.sort_values(["group_id", "Date"]).reset_index(drop=True)

assert "zscore" in df_raw.columns, (
    "'zscore' column missing -- did you rerun the data pipeline with the "
    "patched build_panel from Part 1? The CSV needs to be regenerated first."
)

# Target: next-day change in the VECM spread z-score, per group
df_raw["target"] = df_raw.groupby("group_id")["zscore"].diff().shift(-1)
df_raw = df_raw.dropna(subset=["target"]).reset_index(drop=True)

max_encoder_length = 60
horizon = 5
val_window = 60

def assign_split(g):
    g = g.sort_values("Date").reset_index(drop=True)
    n = len(g)
    g["split"] = "train"
    g.loc[n - horizon:, "split"] = "test"
    g.loc[n - horizon - val_window: n - horizon - 1, "split"] = "val"
    return g

df_raw = df_raw.groupby("group_id", group_keys=False).apply(assign_split)
df = df_raw.copy()

# ── Sanity check ─────────────────────────────────────────────────────────
for split_name in ["train", "val", "test"]:
    sub = df[df["split"] == split_name]
    print(split_name, sub.shape,
          sub["Date"].min().date() if len(sub) else "-", "->",
          sub["Date"].max().date() if len(sub) else "-")
    counts = sub.groupby("group_id").size()
    print("  rows/group: min", counts.min() if len(counts) else 0,
          "max", counts.max() if len(counts) else 0)

assert (df.groupby("group_id").size() >= max_encoder_length + val_window + horizon).all(), \
    "Some group has too few rows for the configured window sizes."

# ── SAVE: full df + each split separately, plus the config used to build
#    them. Anything downstream (or a future session) can reload from here
#    without rerunning the pipeline or the split logic. ────────────────────
df.to_parquet(f"{DATA_DIR}full_dataset.parquet", index=False)
for split_name in ["train", "val", "test"]:
    df[df["split"] == split_name].to_parquet(f"{DATA_DIR}{split_name}.parquet", index=False)

with open(f"{DATA_DIR}split_config.json", "w") as f:
    json.dump({
        "max_encoder_length": max_encoder_length,
        "horizon": horizon,
        "val_window": val_window,
        "csv_source": CSV_PATH,
        "n_groups": int(df["group_id"].nunique()),
        "date_min": str(df["Date"].min().date()),
        "date_max": str(df["Date"].max().date()),
    }, f, indent=2)

print(f"✅ Saved full dataset + train/val/test splits + config to {DATA_DIR}")


/tmp/ipykernel_7152/3208529104.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_raw = df_raw.groupby("group_id", group_keys=False).apply(assign_split)


train (55540, 618) 2013-03-05 -> 2023-10-25
  rows/group: min 2777 max 2777
val (1200, 618) 2023-10-26 -> 2024-01-17
  rows/group: min 60 max 60
test (100, 618) 2024-01-18 -> 2024-01-24
  rows/group: min 5 max 5
✅ Saved full dataset + train/val/test splits + config to /content/drive/MyDrive/fx_models/data/


### Step 1b — (Optional) Reload from saved splits

Only needed if you're resuming a session instead of running Step 1 fresh in this runtime.

In [11]:
df = pd.read_parquet(f"{DATA_DIR}full_dataset.parquet")
train_df = pd.read_parquet(f"{DATA_DIR}train.parquet")
val_df   = pd.read_parquet(f"{DATA_DIR}val.parquet")
test_df  = pd.read_parquet(f"{DATA_DIR}test.parquet")
with open(f"{DATA_DIR}split_config.json") as f:
    cfg = json.load(f)
max_encoder_length, horizon, val_window = cfg["max_encoder_length"], cfg["horizon"], cfg["val_window"]


### Step 2 — Feature column lists shared across all models

In [12]:
# ══════════════════════════════════════════════════════════════════════════
# Cell B — Feature column lists shared across all models  (V6)
# ══════════════════════════════════════════════════════════════════════════
# Four feature families, each handled differently downstream:
#
#   1. gdelt_cols          -- GDELT news/shock features. NOT knowable in
#                              advance. Historical/unknown input only.
#   2. dcc_garch_leg_cols   -- pairwise correlation (rho_) + per-leg
#                              volatility (sigma_), restricted to THIS
#                              group's own 3 legs. NOT knowable in advance.
#   3. macro_leg_cols       -- macroeconomic indicators (growth/inflation/
#                              labor/pmi/rate/trade + surprise component),
#                              restricted to THIS group's own 3 legs. NOT
#                              knowable in advance.
#   4. calendar_cols        -- calendar/seasonality features. KNOWABLE in
#                              advance (day-of-week for a future date is
#                              computable today).
#
# WHY "restricted to this group's own legs": group_id is a 3-pair triplet
# (e.g. "AUDCAD=X_AUDUSD=X_NZDCAD=X" -- confirmed from your diagnostics),
# but the raw macro_*/rho_/sigma_ columns cover all 20 underlying pairs
# (277 macro_* columns alone). Feeding every group's model data about pairs
# it has nothing to do with both dilutes the signal and blows up the
# feature/row ratio against max_encoder_length=60. Per your call: only each
# group's own 3 legs get used, via generic leg1_*/leg2_*/leg3_* columns
# (same column names across every group, different underlying values per
# group's own legs).
# ══════════════════════════════════════════════════════════════════════════

gdelt_cols = [c for c in df.columns if any(
    c.startswith(p) for p in
    ["shock_events_diff", "avg_goldstein_diff", "avg_tone_diff", "mentions_diff", "articles_diff"]
) and not c.endswith(("=X",))]  # group-level aggregates only, not per-pair suffixed dupes

# V6 FIX: tone_shock_interaction / tone_mentions_interaction are GDELT-derived
# interaction terms (tone x shock-count, tone x mentions-count) but don't match
# any of the startswith() prefixes above, so the pattern match alone misses
# them. They're the same "not knowable in advance, group-level" shape as the
# rest of gdelt_cols, so they belong in the same list rather than a separate
# one -- appended explicitly here instead of widening the pattern (which would
# risk catching something unintended).
_gdelt_interaction_cols = ["tone_shock_interaction", "tone_mentions_interaction"]
for col in _gdelt_interaction_cols:
    if col not in df.columns:
        df[col] = np.nan
gdelt_cols = gdelt_cols + [c for c in _gdelt_interaction_cols if c not in gdelt_cols]

_calendar_patterns = [
    "dow", "day_of_week", "weekday", "month", "quarter", "week_of_year",
    "day_of_year", "doy", "is_month_start", "is_month_end",
    "is_quarter_start", "is_quarter_end", "is_year_start", "is_year_end",
    "holiday", "is_holiday", "days_to_holiday", "days_since_holiday",
    "trading_day", "is_weekend", "session",
]
_non_feature_cols = {"target", "split", "group_id", "Date", "time_idx"}
calendar_cols = [
    c for c in df.columns
    if c not in _non_feature_cols and c not in set(gdelt_cols)
    and any(p in c.lower() for p in _calendar_patterns)
]

print(f"gdelt_cols    ({len(gdelt_cols)}):", gdelt_cols[:10], "...")
print(f"calendar_cols ({len(calendar_cols)}) -- treated as KNOWN-FUTURE:", calendar_cols)
if not calendar_cols:
    print("⚠️  No calendar_cols auto-detected -- set manually if your CSV uses different names.")

# ── Parse each group_id into its 3 constituent legs ─────────────────────────
# Currency tickers never contain "_", so splitting the group_id string on
# "_" cleanly recovers the legs (e.g. "AUDCAD=X_AUDUSD=X_NZDCAD=X" ->
# ["AUDCAD=X", "AUDUSD=X", "NZDCAD=X"]).
_group_legs = {g: g.split("_") for g in df["group_id"].unique()}
_bad_groups = {g: legs for g, legs in _group_legs.items() if len(legs) != 3}
if _bad_groups:
    print(f"⚠️  {len(_bad_groups)} group_id value(s) didn't parse into exactly 3 legs -- "
          f"leg-based macro/DCC-GARCH features will be left NaN for these groups: "
          f"{list(_bad_groups)[:5]}")

# ── Macro-economic indicators, restricted to each group's own 3 legs ───────
# LEAK GUARD: macro_<ticker>=X_lead1 / macro_surprise_<ticker>=X_lead1 are
# shifted-forward values -- same leak category as event_lead_1/2, already
# removed elsewhere in this pipeline. Deliberately excluded: "_lead1" never
# appears in the suffix lists below.
_macro_metric_suffixes    = ["", "_growth", "_inflation", "_labor", "_pmi", "_rate", "_trade", "_3d"]
_macro_surprise_suffixes  = ["", "_3d"]

macro_leg_cols = (
    [f"leg{i}_macro{s}" for i in (1, 2, 3) for s in _macro_metric_suffixes]
    + [f"leg{i}_macro_surprise{s}" for i in (1, 2, 3) for s in _macro_surprise_suffixes]
)
for col in macro_leg_cols:
    if col not in df.columns:
        df[col] = np.nan

_missing_macro_src = set()
for g, legs in _group_legs.items():
    if len(legs) != 3:
        continue
    mask = df["group_id"] == g
    for leg_idx, ticker in zip((1, 2, 3), legs):
        for suf in _macro_metric_suffixes:
            src, dst = f"macro_{ticker}{suf}", f"leg{leg_idx}_macro{suf}"
            if src in df.columns:
                df.loc[mask, dst] = df.loc[mask, src].values
            else:
                _missing_macro_src.add(src)
        for suf in _macro_surprise_suffixes:
            src, dst = f"macro_surprise_{ticker}{suf}", f"leg{leg_idx}_macro_surprise{suf}"
            if src in df.columns:
                df.loc[mask, dst] = df.loc[mask, src].values
            else:
                _missing_macro_src.add(src)

if _missing_macro_src:
    print(f"⚠️  {len(_missing_macro_src)} expected macro_* source columns not found -- "
          f"corresponding leg features left NaN: {sorted(_missing_macro_src)[:10]} ...")

print(f"macro_leg_cols ({len(macro_leg_cols)}): built from each group's own 3 legs, "
      f"*_lead1 excluded (leak guard)")

# ── DCC-GARCH rho_/sigma_, restricted to each group's own legs ─────────────
# V6 FIX (confirmed against full 616-column dump):
#   - rho_ columns use BARE ticker names, no "=X" ("rho_AUDCAD_AUDUSD") --
#     unchanged from before, this part was already correct.
#   - sigma_ columns use the ticker WITH "=X" ("sigma_AUDCAD=X"), NOT bare
#     ("sigma_AUDCAD") -- the earlier bare-ticker guess for sigma_ was wrong.
#     Fixed below: sigma_ lookup now uses the raw leg ticker (which already
#     carries "=X"), while rho_ lookup still uses the bare ticker.
#   - rho_ also has _change/_abs variants (e.g. "rho_AUDCAD_AUDUSD_change",
#     "_abs") that were missing from dcc_garch_leg_cols entirely -- same
#     historical-derived treatment as the macro _3d variants above. Added.
def _bare(ticker):
    return ticker.replace("=X", "")

_rho_suffixes = ["", "_change", "_abs"]

dcc_garch_leg_cols = (
    [f"rho_leg{a}_leg{b}{s}" for a, b in [(1, 2), (1, 3), (2, 3)] for s in _rho_suffixes]
    + [f"sigma_leg{i}" for i in (1, 2, 3)]
)
for col in dcc_garch_leg_cols:
    if col not in df.columns:
        df[col] = np.nan

_missing_dcc_src = set()
for g, legs in _group_legs.items():
    if len(legs) != 3:
        continue
    mask = df["group_id"] == g
    bare = [_bare(t) for t in legs]

    for (a, b), (ia, ib) in zip([(1, 2), (1, 3), (2, 3)], [(0, 1), (0, 2), (1, 2)]):
        for suf in _rho_suffixes:
            dst = f"rho_leg{a}_leg{b}{suf}"
            candidates = [f"rho_{bare[ia]}_{bare[ib]}{suf}", f"rho_{bare[ib]}_{bare[ia]}{suf}"]
            src = next((c for c in candidates if c in df.columns), None)
            if src:
                df.loc[mask, dst] = df.loc[mask, src].values
            else:
                _missing_dcc_src.update(candidates)

    # sigma_ uses the ticker WITH "=X" (e.g. "sigma_AUDCAD=X"), not bare --
    # use the raw leg ticker directly, not the bare-stripped version.
    for leg_idx, ticker in zip((1, 2, 3), legs):
        dst, src = f"sigma_leg{leg_idx}", f"sigma_{ticker}"
        if src in df.columns:
            df.loc[mask, dst] = df.loc[mask, src].values
        else:
            _missing_dcc_src.add(src)

if _missing_dcc_src:
    print(f"⚠️  {len(_missing_dcc_src)} expected rho_/sigma_ source columns not found -- "
          f"corresponding leg features left NaN: {sorted(_missing_dcc_src)[:10]} ...")

print(f"dcc_garch_leg_cols ({len(dcc_garch_leg_cols)}): pairwise rho_ (+ _change/_abs) + "
      f"per-leg sigma_, restricted to each group's own legs")

# ── V6 ADD: regime labels + global_macro_cols + event_flag ────────────────
# regime/regime_vol/regime_macro were sitting in the source CSV but reaching
# neither model consistently -- NeuralForecast (Cell I) was factorizing them
# locally, but TFT (Cell C) never received them at all. Moved the
# factorization up here so both models train on the identical encoding
# instead of two independent pd.factorize() calls (which is not guaranteed
# to assign the same integer codes to the same category across two separate
# calls, even on the same underlying values).
for _col in ["regime", "regime_vol", "regime_macro"]:
    df[f"{_col}_num"] = pd.factorize(df[_col])[0].astype(float)
regime_cols = ["regime_num", "regime_vol_num", "regime_macro_num"]

# global_macro_cols: macro_pressure is a SINGLE global macro-pressure summary
# -- unlike macro_leg_cols (per-group, restricted to each group's own 3 legs)
# and unlike the pairwise rho_/sigma_ family, this one column is the same
# value for every group_id on a given date. Kept as its own explicitly-named
# list rather than folded into gdelt_cols or macro_leg_cols, since it isn't
# GDELT-derived and isn't per-leg -- the distinct name makes that "one global
# column, not per-pair" shape visible at the call site instead of hidden
# inside a same-named list as things that ARE per-pair. NOT knowable in
# advance -- historical-only input for both models.
global_macro_cols = ["macro_pressure"]
for col in global_macro_cols:
    if col not in df.columns:
        df[col] = np.nan

# event_flag: same-day GDELT shock flag -- same leak category as
# event_lag_1/2 (already historical-only in both models). Was referenced in
# the V4 changelog note as "should be historical only" but never actually
# added to either model's feature list until now.
extra_event_cols = ["event_flag"]
for col in extra_event_cols:
    if col not in df.columns:
        df[col] = np.nan

print(f"regime_cols ({len(regime_cols)}) + global_macro_cols ({len(global_macro_cols)}) + "
      f"extra_event_cols ({len(extra_event_cols)}): now wired into both TFT and NeuralForecast, "
      f"historical-only. tone_shock_interaction/tone_mentions_interaction folded into "
      f"gdelt_cols above -- gdelt_cols is now ({len(gdelt_cols)}) columns.")

unknown_cols = (
    ["target", "event_lag_1", "event_lag_2"] + extra_event_cols
    + gdelt_cols + dcc_garch_leg_cols + macro_leg_cols + regime_cols + global_macro_cols
)
known_cols   = calendar_cols
static_cols  = []

# Manual override point -- uncomment and edit if any auto-detection above is wrong:
# calendar_cols = ["dow_sin", "dow_cos", "month_sin", "month_cos", "is_holiday"]


gdelt_cols    (47): ['shock_events_diff', 'avg_goldstein_diff', 'avg_tone_diff', 'mentions_diff', 'articles_diff', 'shock_events_diff_mean_3', 'shock_events_diff_std_3', 'shock_events_diff_mean_7', 'shock_events_diff_std_7', 'shock_events_diff_mean_14'] ...
calendar_cols (0) -- treated as KNOWN-FUTURE: []
⚠️  No calendar_cols auto-detected -- set manually if your CSV uses different names.
macro_leg_cols (30): built from each group's own 3 legs, *_lead1 excluded (leak guard)
dcc_garch_leg_cols (12): pairwise rho_ (+ _change/_abs) + per-leg sigma_, restricted to each group's own legs
regime_cols (3) + global_macro_cols (1) + extra_event_cols (1): now wired into both TFT and NeuralForecast, historical-only. tone_shock_interaction/tone_mentions_interaction folded into gdelt_cols above -- gdelt_cols is now (47) columns.


In [13]:
# ── NaN-rate check for dcc_garch_leg_cols + macro_leg_cols ─────────────────
# Run this right after Cell B, once df has dcc_garch_leg_cols/macro_leg_cols
# (and regime_cols/global_macro_cols/extra_event_cols) all built.

nan_check_cols = dcc_garch_leg_cols + macro_leg_cols
nan_rates = df[nan_check_cols].isna().mean().sort_values(ascending=False)

print("Top 15 highest-NaN columns:")
print(nan_rates.head(15))
print()
print(f"Overall mean NaN rate across these {len(nan_check_cols)} columns: {nan_rates.mean():.2%}")
print(f"Columns >20% NaN: {(nan_rates > 0.20).sum()} / {len(nan_check_cols)}")
print(f"Columns >50% NaN: {(nan_rates > 0.50).sum()} / {len(nan_check_cols)}")

# Also worth checking per-group, since dcc_garch_leg_cols/macro_leg_cols are
# built per group_id -- a group with a bad _group_legs parse (see the
# "didn't parse into exactly 3 legs" warning earlier in Cell B) will show
# 100% NaN for that group specifically, which the aggregate mean above can
# mask if most other groups are fine.
nan_by_group = (
    df.groupby("group_id")[nan_check_cols]
    .apply(lambda g: g.isna().mean().mean())
    .sort_values(ascending=False)
)
print()
print("Worst 10 groups by mean NaN rate (across dcc_garch_leg_cols + macro_leg_cols):")
print(nan_by_group.head(10))

Top 15 highest-NaN columns:
rho_leg1_leg2           0.0
rho_leg1_leg2_change    0.0
rho_leg1_leg2_abs       0.0
rho_leg1_leg3           0.0
rho_leg1_leg3_change    0.0
rho_leg1_leg3_abs       0.0
rho_leg2_leg3           0.0
rho_leg2_leg3_change    0.0
rho_leg2_leg3_abs       0.0
sigma_leg1              0.0
sigma_leg2              0.0
sigma_leg3              0.0
leg1_macro              0.0
leg1_macro_growth       0.0
leg1_macro_inflation    0.0
dtype: float64

Overall mean NaN rate across these 42 columns: 0.00%
Columns >20% NaN: 0 / 42
Columns >50% NaN: 0 / 42

Worst 10 groups by mean NaN rate (across dcc_garch_leg_cols + macro_leg_cols):
group_id
AUDCAD=X_AUDUSD=X_NZDCAD=X    0.0
AUDCAD=X_NZDCAD=X_NZDUSD=X    0.0
AUDCHF=X_EURAUD=X_EURNZD=X    0.0
AUDCHF=X_EURAUD=X_NZDCHF=X    0.0
AUDCHF=X_EURNZD=X_NZDCHF=X    0.0
AUDJPY=X_AUDUSD=X_NZDUSD=X    0.0
AUDJPY=X_CADJPY=X_NZDJPY=X    0.0
AUDJPY=X_NZDCHF=X_NZDJPY=X    0.0
AUDNZD=X_EURNZD=X_NZDCAD=X    0.0
AUDUSD=X_CADJPY=X_USDCAD=X    0.0
dtyp

Clean file

In [14]:
# ── Full-folder reset before a clean rerun ──────────────────────────────────
# Clears every model-output subfolder under drive_base so a rerun starts from
# nothing instead of accumulating old checkpoints/predictions/logs alongside
# new ones. Run this right after Cell 0 (Step 0b, which defines drive_base
# and recreates these folders), before Step 1.

import shutil, os

# Model-output folders only -- NOT DATA_DIR. DATA_DIR holds train/val/test
# parquet splits + split_config.json, which are expensive to regenerate and
# which Step 1b is specifically built to reload -- wiping those on every
# rerun would silently force a fresh chronological split each time, which is
# a much bigger change than "clean model artifacts."
CLEAR_SUBFOLDERS = ["tft", "nbeats", "nhits", "patchtst", "spacetimeformer", "ensemble", "leaderboard"]

# Set this True only if you also want DATA_DIR wiped -- i.e. you intend to
# rebuild the splits from scratch this run, not just retrain models.
ALSO_CLEAR_DATA_DIR = False

for sub in CLEAR_SUBFOLDERS:
    path = f"{drive_base}{sub}/"
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
    print(f"Cleared {path}")

if ALSO_CLEAR_DATA_DIR and os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"Cleared {DATA_DIR} (ALSO_CLEAR_DATA_DIR=True)")
elif not ALSO_CLEAR_DATA_DIR:
    print(f"Left {DATA_DIR} untouched (ALSO_CLEAR_DATA_DIR=False) -- "
          f"train/val/test splits will be reused if present.")

# manifest.json / hyperparameters.json live directly under drive_base, not
# inside a subfolder -- these get overwritten (not accumulated) by Step 12
# regardless, so no separate clear needed for them.

Cleared /content/drive/MyDrive/fx_models/tft/
Cleared /content/drive/MyDrive/fx_models/nbeats/
Cleared /content/drive/MyDrive/fx_models/nhits/
Cleared /content/drive/MyDrive/fx_models/patchtst/
Cleared /content/drive/MyDrive/fx_models/spacetimeformer/
Cleared /content/drive/MyDrive/fx_models/ensemble/
Cleared /content/drive/MyDrive/fx_models/leaderboard/
Left /content/drive/MyDrive/fx_models/data/ untouched (ALSO_CLEAR_DATA_DIR=False) -- train/val/test splits will be reused if present.


### Step 3 — Train the Temporal Fusion Transformer (TFT)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell C — TFT: TimeSeriesDataSet + training + SAVE checkpoint + dataset params
# ══════════════════════════════════════════════════════════════════════════
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger

tft_df = df.copy()
tft_df["time_idx"] = tft_df.groupby("group_id").cumcount()

train_val_df = tft_df[tft_df["split"].isin(["train", "val"])]

# The cutoff below is a single global time_idx applied to every group. That
# is only correct if every group's train+val slice has the same length /
# alignment (true here because all pairs share one global START_DATE/
# END_DATE and the panel is built on a common date grid) -- but it is an
# assumption, not something the code previously checked. Verify it before
# trusting the split.
per_group_max = train_val_df.groupby("group_id")["time_idx"].max()
assert per_group_max.nunique() == 1, (
    "Groups have different train+val lengths -- the global training_cutoff "
    "below will not correspond to 'the last val_window rows' for every "
    "group. Fix upstream (or switch to a per-group cutoff) before training."
)
training_cutoff = train_val_df["time_idx"].max() - val_window

# LEAKAGE FIX (unchanged from V3): event_lag_1 stays in unknown_reals, not
# known_reals. "known reals" are treated by pytorch-forecasting as available
# for every step of the prediction horizon -- but event_lag_1 (yesterday's
# GDELT shock flag) is only genuinely known one step ahead. Marking it
# "known" would hand the model real historical values that would not
# actually be available yet at forecast time -- same leak category as
# event_lead_1/2, already removed upstream.
#
# V4 FIX: calendar_cols is a categorically different case from event_lag_1.
# Day-of-week, month, quarter, and holiday flags for a FUTURE date are
# genuinely computable today -- that's the entire point of calling them
# "calendar" features. V3 left time_varying_known_reals empty, which meant
# calendar data (despite being in the source CSV) never reached TFT at all.
# calendar_cols now goes into known_reals; gdelt_cols/dcc_garch_leg_cols/
# macro_leg_cols (truly not knowable in advance) stay in unknown_reals
# alongside target/event_lag_1/event_lag_2.
#
# V5 FIX: event_lag_2 exists in the source data but was never used anywhere
# in V3/V4 (only event_lag_1 was wired in) -- added alongside it, same
# unknown/historical treatment. gdelt_cols' old blanket "macro_cols" (rho_/
# sigma_ only) is replaced by dcc_garch_leg_cols + macro_leg_cols, which
# also pull in the 277-column macro_* family V4 missed entirely -- see the
# V5 changelog cell at the top of the notebook.
training_dataset = TimeSeriesDataSet(
    train_val_df[train_val_df["time_idx"] <= training_cutoff],
    time_idx="time_idx",
    target="target",
    group_ids=["group_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=horizon,
    time_varying_known_reals=calendar_cols,
    time_varying_unknown_reals=["target", "event_lag_1", "event_lag_2"] + extra_event_cols
        + gdelt_cols + dcc_garch_leg_cols + macro_leg_cols + regime_cols + global_macro_cols,
    target_normalizer=GroupNormalizer(groups=["group_id"]),
    add_relative_time_idx=True,
    add_target_scales=True,
)
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, train_val_df, predict=True, stop_randomization=True
)

# SAVE dataset parameters -- required to reconstruct a TimeSeriesDataSet
# for inference later without needing the original training dataframe.
with open(f"{drive_base}tft/dataset_params.pkl", "wb") as f:
    pickle.dump(training_dataset.get_parameters(), f)

train_loader = training_dataset.to_dataloader(train=True, batch_size=64, num_workers=2)
val_loader   = validation_dataset.to_dataloader(train=False, batch_size=64, num_workers=2)

checkpoint_cb = ModelCheckpoint(
    dirpath=f"{drive_base}tft/checkpoints/",
    filename="best-{epoch}-{val_loss:.4f}",
    monitor="val_loss", mode="min", save_top_k=1,
)
early_stop_cb = EarlyStopping(monitor="val_loss", patience=5, mode="min")

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=1e-3, hidden_size=32, attention_head_size=4,
    dropout=0.1, hidden_continuous_size=16,
    loss=QuantileLoss(), log_interval=10, reduce_on_plateau_patience=3,
)

trainer = pl.Trainer(
    max_epochs=50,
    callbacks=[checkpoint_cb, early_stop_cb, LearningRateMonitor()],
    logger=CSVLogger(save_dir=f"{drive_base}tft/", name="logs"),
    gradient_clip_val=0.1,
)
trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)

best_tft_path = checkpoint_cb.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_tft_path)
print(f"✅ Best TFT checkpoint saved at: {best_tft_path}")

# ── Record hyperparameters for the consolidated run record (Step 12) ──────
tft_hparams = {
    "model": "TFT",
    "learning_rate": 1e-3,
    "hidden_size": 32,
    "attention_head_size": 4,
    "dropout": 0.1,
    "hidden_continuous_size": 16,
    "loss": "QuantileLoss",
    "max_epochs": 50,
    "gradient_clip_val": 0.1,
    "early_stop_patience": 5,
    "max_encoder_length": max_encoder_length,
    "max_prediction_length": horizon,
    "val_window": val_window,
    "time_varying_known_reals": calendar_cols,
    "time_varying_unknown_reals": ["target", "event_lag_1", "event_lag_2"] + extra_event_cols
        + gdelt_cols + dcc_garch_leg_cols + macro_leg_cols + regime_cols + global_macro_cols,
    "seed": SEED,
}
with open(f"{drive_base}tft/hyperparameters.json", "w") as f:
    json.dump(tft_hparams, f, indent=2)



Epoch 8/49 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 728/847 0:23:57 • 0:03:51 0.52it/s v_num: 0.000 train_loss_step:     
                                                                                 0.041 val_loss: 0.026             
                                                                                 train_loss_epoch: 0.045           

### Step 4 — Evaluate TFT on the true held-out test window

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell D — TFT: evaluate on true held-out test window, SAVE metrics + preds
# ══════════════════════════════════════════════════════════════════════════
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, tft_df, predict=True, stop_randomization=True
)
test_loader = test_dataset.to_dataloader(train=False, batch_size=64, num_workers=2)

test_metrics = trainer.test(best_tft, dataloaders=test_loader, verbose=True)

tft_preds = best_tft.predict(test_loader, mode="prediction", return_x=True)
with open(f"{drive_base}tft/test_predictions.pkl", "wb") as f:
    pickle.dump(tft_preds, f)
with open(f"{drive_base}tft/test_metrics.pkl", "wb") as f:
    pickle.dump(test_metrics, f)

print(f"✅ TFT test metrics + predictions saved to {drive_base}tft/")


### Step 5 — Shared helper: exact quantile-column lookup

Used by every NeuralForecast-family model below (NBEATSx, NHITS, PatchTST) so quantile columns are
matched exactly instead of via fragile substring checks like `"50" in c`.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Shared helper
# Exact quantile-column lookup instead of substring guessing. NeuralForecast
# names quantile columns as f"{model_name}-lo-{level}" / f"{model_name}-hi-
# {level}" / f"{model_name}-median" when you pass a MQLoss with `quantiles=`.
# We build the expected names directly from the quantiles the model was
# actually given, so there's no ambiguity.
# ──────────────────────────────────────────────────────────────────────────
def pick_quantile_cols(preds_df: pd.DataFrame, model_name: str, quantiles=(0.1, 0.5, 0.9)):
    """
    Return {quantile: column_name} for a NeuralForecast predictions frame,
    verifying the columns actually exist instead of guessing by substring.
    """
    cols = {}
    candidates = [c for c in preds_df.columns if c.startswith(model_name)]

    for q in quantiles:
        if abs(q - 0.5) < 1e-9:
            match = [c for c in candidates if c == f"{model_name}-median" or c.endswith("-median")]
        else:
            pct = round(q * 100)
            match = [c for c in candidates if c.endswith(f"-{pct}") or c.endswith(f"-{pct}.0")]
        if not match:
            raise ValueError(
                f"Could not find a column for quantile {q} in {model_name} predictions. "
                f"Available columns: {candidates}"
            )
        cols[q] = match[0]

    missing = set(quantiles) - set(cols)
    if missing:
        raise ValueError(f"Missing quantile columns {missing} for {model_name}: {candidates}")
    return cols


### Step 6 — Train NBEATSx, NHITS, PatchTST (NeuralForecast)

`val_size` is fixed to `val_window` (not `val_window + horizon`) so the internal validation window
matches the declared val split instead of silently eating `horizon` extra days out of train.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell I — NeuralForecast models, val_size FIXED, future-exog leak FIXED
# ──────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import LearningRateMonitor
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx, NHITS, PatchTST
from neuralforecast.losses.pytorch import MQLoss, MAE

# V6 FIX: regime_num/regime_vol_num/regime_macro_num are now computed
# once in Cell B (shared with TFT) instead of a second, independent
# pd.factorize() call here -- two separate factorize() calls are not
# guaranteed to assign the same integer codes to the same category, so the
# old local version risked TFT and NeuralForecast training on different
# encodings of the same regime labels. Pulled straight from df below.
# V6 ADD: extra_event_cols (event_flag) and global_macro_cols (macro_pressure)
# were in the source CSV but reached neither model before -- added alongside
# the existing historical-only features. tone_shock_interaction/
# tone_mentions_interaction are already inside gdelt_cols (folded in back in
# Cell B), not listed separately here.
nf_df = df[[
    "group_id", "Date", "target", "split",
    "event_lag_1", "event_lag_2",
] + extra_event_cols + regime_cols + global_macro_cols
  + gdelt_cols + dcc_garch_leg_cols + macro_leg_cols + calendar_cols].copy()

nf_df = nf_df.rename(columns={"group_id": "unique_id", "Date": "ds", "target": "y"})
nf_df["ds"] = pd.to_datetime(nf_df["ds"])

# train + val only -- ends exactly at the start of the true test window
nf_train = nf_df[nf_df["split"].isin(["train", "val"])].drop(columns=["split"])

# LEAKAGE FIX (unchanged from V3): futr_exog_list must never contain
# same-day/unlagged GDELT data. "future exog" is treated by NeuralForecast
# as genuinely known for every future step of the horizon -- GDELT shocks
# are not knowable in advance (same reasoning already applied when the data
# pipeline dropped event_lead_1/2). The lagged event_lag_1 (one-step-old,
# genuinely historical) stays in hist_exog_list, alongside GDELT/macro/
# regime features.
#
# V4 FIX (feature parity with TFT): calendar_cols is genuinely known for
# every future step -- day-of-week/month/quarter/holiday flags for a future
# date are computable today. V3 left futr_exog_list empty, so calendar data
# never reached NBEATSx/NHITS/PatchTST at all. It now goes into
# futr_exog_list, matching how TFT uses time_varying_known_reals.
#
# V4 FIX (feature parity with TFT, #2): V3's hist_exog_list omitted the
# DCC-GARCH columns entirely -- only TFT received them. NBEATSx/NHITS/
# PatchTST now get the same historical exogenous features TFT gets.
#
# V5 FIX: replaced the old blanket "macro_cols" (rho_/sigma_ only) with
# dcc_garch_leg_cols + macro_leg_cols -- restricted to each group's own 3
# legs (not all 20 pairs), and now also includes the 277-column macro_*
# economic-indicator family V4 missed. event_lag_2 added alongside
# event_lag_1 (present in the data, never wired in before).
hist_exog = (
    gdelt_cols + dcc_garch_leg_cols + macro_leg_cols + regime_cols + global_macro_cols
    + ["event_lag_1", "event_lag_2"] + extra_event_cols
)
common_kwargs = dict(
    h=horizon,
    input_size=max_encoder_length,
    futr_exog_list=calendar_cols,
    hist_exog_list=hist_exog,
    loss=MQLoss(quantiles=[0.1, 0.5, 0.9]),
    valid_loss=MAE(),
    max_steps=300,
    val_check_steps=50,
    early_stop_patience_steps=5,
    scaler_type="standard",
    random_seed=SEED,
)

nbeats_model = NBEATSx(
    stack_types=["trend", "seasonality", "identity"],
    n_blocks=[3, 3, 3],
    mlp_units=[[256, 256]] * 3,
    dropout_prob_theta=0.1,
    callbacks=[LearningRateMonitor(logging_interval="epoch")],
    trainer_kwargs={"logger": CSVLogger(save_dir=f"{drive_base}nbeats/", name="logs", version="")},
    **common_kwargs,
)
nhits_model = NHITS(
    n_freq_downsample=[4, 2, 1],
    n_blocks=[3, 3, 3],
    mlp_units=[[256, 256]] * 3,
    callbacks=[LearningRateMonitor(logging_interval="epoch")],
    trainer_kwargs={"logger": CSVLogger(save_dir=f"{drive_base}nhits/", name="logs", version="")},
    **common_kwargs,
)
patchtst_model = PatchTST(
    patch_len=6, stride=1, d_model=128, n_heads=4, e_layers=3, dropout=0.1,
    callbacks=[LearningRateMonitor(logging_interval="epoch")],
    trainer_kwargs={"logger": CSVLogger(save_dir=f"{drive_base}patchtst/", name="logs", version="")},
    **common_kwargs,
)

nf_nbeats   = NeuralForecast(models=[nbeats_model],   freq="B")
nf_nhits    = NeuralForecast(models=[nhits_model],    freq="B")
nf_patchtst = NeuralForecast(models=[patchtst_model], freq="B")

# val_size = val_window (not val_window + horizon). nf_train already stops
# exactly at the start of the test window, so carving off val_window from
# ITS tail reproduces the declared val split exactly -- no extra days
# borrowed from train, and TFT/NF now validate on the same-length window.
nf_nbeats.fit(nf_train, val_size=val_window)
nbeats_preds = nf_nbeats.predict()

nf_nhits.fit(nf_train, val_size=val_window)
nhits_preds = nf_nhits.predict()

nf_patchtst.fit(nf_train, val_size=val_window)
patchtst_preds = nf_patchtst.predict()

for name, nf_obj in [("nbeats", nf_nbeats), ("nhits", nf_nhits), ("patchtst", nf_patchtst)]:
    os.makedirs(f"{drive_base}{name}/", exist_ok=True)
    nf_obj.save(f"{drive_base}{name}/", overwrite=True)

# save raw prediction frames (before test scoring) for later inspection
nbeats_preds.to_csv(f"{drive_base}nbeats/test_predictions.csv", index=False)
nhits_preds.to_csv(f"{drive_base}nhits/test_predictions.csv", index=False)
patchtst_preds.to_csv(f"{drive_base}patchtst/test_predictions.csv", index=False)

print("✅ NBEATSx / NHITS / PatchTST trained on train+val, validated on the "
      "declared val_window (no more borrowed train days), no future-exog leak")

# ── Record hyperparameters for the consolidated run record (Step 12) ──────
nf_hparams = {
    "shared": {
        "h": horizon,
        "input_size": max_encoder_length,
        "futr_exog_list": calendar_cols,
        "hist_exog_list": hist_exog,
        "loss": "MQLoss(quantiles=[0.1, 0.5, 0.9])",
        "valid_loss": "MAE",
        "max_steps": 300,
        "val_check_steps": 50,
        "early_stop_patience_steps": 5,
        "scaler_type": "standard",
        "val_size": val_window,
        "seed": SEED,
    },
    "NBEATSx": {
        "stack_types": ["trend", "seasonality", "identity"],
        "n_blocks": [3, 3, 3],
        "mlp_units": [[256, 256]] * 3,
        "dropout_prob_theta": 0.1,
    },
    "NHITS": {
        "n_freq_downsample": [4, 2, 1],
        "n_blocks": [3, 3, 3],
        "mlp_units": [[256, 256]] * 3,
    },
    "PatchTST": {
        "patch_len": 6, "stride": 1, "d_model": 128, "n_heads": 4,
        "e_layers": 3, "dropout": 0.1,
    },
}
for name in ["nbeats", "nhits", "patchtst"]:
    with open(f"{drive_base}{name}/hyperparameters.json", "w") as f:
        json.dump(nf_hparams, f, indent=2)



### Step 7 — Train Spacetimeformer (train+val only)

Built from train+val only, exactly like `nf_train`, so its "val" slice can never contain true test
rows. Checkpoints on best `val_loss` and loads that checkpoint before saving / evaluating.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell J — Spacetimeformer, train+val ONLY   (V4)
# ──────────────────────────────────────────────────────────────────────────
# WHY THIS ISN'T A CLEAN "ADD gdelt_cols + calendar_cols" LIKE TFT/NF:
#
# TFT and the NeuralForecast models already accept named exogenous-feature
# lists -- adding calendar/GDELT there is wiring, not architecture change.
# Spacetimeformer here is different. The class actually used --
# `stf.model.Spacetimeformer(d_x=n_pairs, d_y=n_pairs, max_seq_len=...,
# out_len=...)` with a single-tensor `forward(X)` call -- does NOT match the
# published `spacetimeformer` (QData) library's real API, which is
# `Spacetimeformer_Forecaster` taking separate `x_context, y_context,
# x_target, y_target` tensors. That means either a custom/simplified wrapper
# is installed in this environment, or there's a version mismatch -- and I
# could not confirm which from inside this notebook (it's installed outside
# Cell 2's `!pip install` line, so its actual signature is opaque here).
#
# CONCRETELY: does the class in *your* environment support d_x != d_y (more
# input channels -- target + exogenous -- than output channels -- target
# only)? I don't know, and guessing wrong here either crashes (safe) or
# silently trains on a broadcast/reshape that doesn't mean what you think it
# means (not safe, and much worse than a crash).
#
# BEFORE flipping STF_USE_EXOG to True below, run this in your Colab
# (wherever `stf` gets imported) and read the output:
#
#     import spacetimeformer as stf
#     print(stf.__file__)
#     help(stf.model.Spacetimeformer.__init__)
#     help(stf.model.Spacetimeformer.forward)
#
# Confirm d_x != d_y is accepted and that forward() takes a single tensor
# shaped (batch, seq_len, d_x) and returns (batch, out_len, d_y). If either
# of those isn't true, leave STF_USE_EXOG = False -- the try/except below
# will also auto-fall-back to the autoregressive path if construction or a
# forward pass fails, but that's a safety net, not a substitute for reading
# the actual signature first.
STF_USE_EXOG = False   # flip to True only after confirming the API above

try:
    import spacetimeformer as stf
    STF_AVAILABLE = True
except ImportError:
    STF_AVAILABLE = False
    print("⚠️  Spacetimeformer not installed — skipping")

stf_model = None
stf_test_mae = None
stf_pair_names = None
stf_exog_cols = None   # None => pure autoregressive; list => exogenous channels used

if STF_AVAILABLE:
    # Build the pivot from train+val ONLY, exactly matching nf_train's
    # scope. The true test window (last `horizon` days) is never touched
    # here -- it's reserved for Step 8's one-time eval.
    df_trainval = df[df["split"].isin(["train", "val"])]
    stf_df = df_trainval.pivot_table(
        index="Date", columns="group_id", values="target"
    ).dropna(how="any")
    stf_pair_names = list(stf_df.columns)
    n_pairs = len(stf_pair_names)

    # ── Exogenous channel assembly (only attempted if STF_USE_EXOG) ────────
    # gdelt_cols/macro_cols/calendar_cols are pulled from Cell B. They only
    # concat cleanly onto this pivot if they're constant across group_id for
    # a given date (i.e. genuinely date-level, not per-pair) -- gdelt_cols
    # was already filtered to "group-level aggregates only" in Cell B, but
    # macro_cols (rho_/sigma_ DCC-GARCH) is NOT guaranteed to be pair-
    # invariant (DCC-GARCH correlations are often pairwise by construction).
    # Check this explicitly instead of assuming it -- silently averaging or
    # taking-first over a genuinely per-pair signal would be its own leak/
    # distortion, quietly wrong in a way that wouldn't show up as a crash.
    # V5: candidate_exog_cols now uses dcc_garch_leg_cols/macro_leg_cols
    # (each group's own 3 legs) instead of the old blanket macro_cols. Note
    # this makes the per-date-nunique check below trigger EVEN MORE
    # reliably now, by design: leg1_macro etc. are deliberately different
    # per group (that's the whole point of the V5 per-group leg fix), so
    # they will always vary by group_id within a date -- meaning STF_USE_EXOG
    # will correctly and automatically fall back to autoregressive-only for
    # these columns unless this notebook is restructured to a real 3D
    # (date x pair x feature) tensor. calendar_cols/gdelt_cols remain
    # genuinely date-level (constant across groups) and could still work
    # with the simple concat path on their own.
    candidate_exog_cols = calendar_cols + gdelt_cols + dcc_garch_leg_cols + macro_leg_cols
    exog_ok = True
    if STF_USE_EXOG and candidate_exog_cols:
        per_date_nunique = (
            df_trainval.groupby("Date")[candidate_exog_cols]
            .nunique()
            .max()
        )
        non_constant = per_date_nunique[per_date_nunique > 1].index.tolist()
        if non_constant:
            print(f"⚠️  These columns vary by group_id within a single date, so they "
                  f"can't be safely concatenated onto the (date x pair) target pivot "
                  f"without a 3D (date x pair x feature) restructure this notebook "
                  f"doesn't do: {non_constant[:10]}{'...' if len(non_constant) > 10 else ''}\n"
                  f"    Falling back to pure-autoregressive Spacetimeformer (no exog).")
            exog_ok = False

    use_exog = STF_USE_EXOG and exog_ok and bool(candidate_exog_cols)

    if use_exog:
        exog_df = (
            df_trainval.drop_duplicates("Date")
            .set_index("Date")[candidate_exog_cols]
            .reindex(stf_df.index)
        )
        n_exog = exog_df.shape[1]
        print(f"Spacetimeformer (EXOG mode): {n_exog} exogenous channels "
              f"({len(calendar_cols)} calendar + {len(gdelt_cols)} gdelt + "
              f"{len(dcc_garch_leg_cols)} dcc_garch + {len(macro_leg_cols)} macro) "
              f"appended to {n_pairs} target pairs.")
    else:
        n_exog = 0
        exog_df = None
        if STF_USE_EXOG and not candidate_exog_cols:
            print("⚠️  STF_USE_EXOG=True but no calendar/gdelt/macro columns were found "
                  "-- nothing to add, using pure-autoregressive Spacetimeformer.")

    print(f"Spacetimeformer training on {len(stf_df)} dates (train+val only), "
          f"{n_pairs} pairs ({stf_df.index.min().date()} -> {stf_df.index.max().date()})"
          + (f", {n_exog} exog channels" if use_exog else " [pure autoregressive]"))

    d_x = n_pairs + n_exog
    stf_arch_hparams = dict(
        d_x=d_x, d_y=n_pairs, max_seq_len=max_encoder_length, out_len=horizon,
        d_model=64, n_heads=4, e_layers=2, d_layers=1, dropout=0.1,
        embed="spatio-temporal", activation="gelu",
    )

    def _build_and_check():
        m = stf.model.Spacetimeformer(**stf_arch_hparams)
        probe = torch.zeros(1, max_encoder_length, d_x)
        with torch.no_grad():
            probe_out = m(probe)
        if probe_out.shape[-1] != n_pairs:
            raise RuntimeError(
                f"Spacetimeformer output last-dim {probe_out.shape[-1]} != n_pairs "
                f"{n_pairs} with d_x={d_x} != d_y={n_pairs} -- this class does not "
                f"support asymmetric d_x/d_y the way this notebook is assuming."
            )
        return m

    if use_exog:
        try:
            stf_model = _build_and_check()
            stf_exog_cols = candidate_exog_cols
        except Exception as e:
            print(f"⚠️  Exogenous Spacetimeformer construction/forward failed: {e}\n"
                  f"    Falling back to pure-autoregressive Spacetimeformer (d_x=d_y=n_pairs).")
            use_exog = False

    if not use_exog:
        d_x = n_pairs
        stf_arch_hparams = dict(
            d_x=n_pairs, d_y=n_pairs, max_seq_len=max_encoder_length, out_len=horizon,
            d_model=64, n_heads=4, e_layers=2, d_layers=1, dropout=0.1,
            embed="spatio-temporal", activation="gelu",
        )
        stf_model = stf.model.Spacetimeformer(**stf_arch_hparams)
        stf_exog_cols = None

    # ── Assemble X/y tensors ────────────────────────────────────────────────
    target_arr = stf_df.values                       # (dates, n_pairs)
    if use_exog:
        exog_arr = exog_df.values                    # (dates, n_exog)
        full_arr = np.concatenate([target_arr, exog_arr], axis=1)  # (dates, d_x)
    else:
        full_arr = target_arr

    X_full = full_arr[:-horizon]
    y_full = target_arr[horizon:]     # y is ALWAYS target-only (d_y = n_pairs)

    # train+val was already carved to exclude the true test window
    # entirely, so val_window is the only split needed here.
    split_point = X_full.shape[0] - val_window

    X_train = torch.tensor(X_full[:split_point], dtype=torch.float32).unsqueeze(0)
    y_train = torch.tensor(y_full[:split_point], dtype=torch.float32).unsqueeze(0)
    X_val   = torch.tensor(X_full[split_point:], dtype=torch.float32).unsqueeze(0)
    y_val   = torch.tensor(y_full[split_point:], dtype=torch.float32).unsqueeze(0)

    stf_train_hparams = {"lr": 1e-3, "n_epochs": 20, "val_window": val_window, "seed": SEED}
    optimizer = torch.optim.Adam(stf_model.parameters(), lr=stf_train_hparams["lr"])
    n_epochs = stf_train_hparams["n_epochs"]
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    mae_fn, mse_fn = nn.L1Loss(), nn.MSELoss()
    stf_history = {"epoch": [], "train_loss": [], "val_loss": [], "train_mae": [], "val_mae": [], "lr": []}

    # Track and checkpoint the best val_loss epoch instead of just keeping
    # whatever weights exist after the fixed epoch count.
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(n_epochs):
        stf_model.train()
        optimizer.zero_grad()
        out = stf_model(X_train)
        y_train_aligned = y_train[:, :out.shape[1], :]
        train_loss = mse_fn(out, y_train_aligned)
        train_mae = mae_fn(out, y_train_aligned)
        train_loss.backward()
        optimizer.step()

        stf_model.eval()
        with torch.no_grad():
            val_out = stf_model(X_val)
            y_val_aligned = y_val[:, :val_out.shape[1], :]
            val_loss = mse_fn(val_out, y_val_aligned)
            val_mae = mae_fn(val_out, y_val_aligned)

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_state = {k: v.detach().clone() for k, v in stf_model.state_dict().items()}

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()
        for k, v in [("epoch", epoch), ("train_loss", train_loss.item()), ("val_loss", val_loss.item()),
                     ("train_mae", train_mae.item()), ("val_mae", val_mae.item()), ("lr", current_lr)]:
            stf_history[k].append(v)

        if epoch % 5 == 0:
            print(f"  Epoch {epoch:3d} | LR {current_lr:.6f} | "
                  f"train_loss {train_loss.item():.6f} | val_loss {val_loss.item():.6f}")

    # load best-val-loss weights before saving / evaluating
    if best_state is not None:
        stf_model.load_state_dict(best_state)
        print(f"✅ Loaded best checkpoint (val_loss={best_val_loss:.6f})")

    os.makedirs(f"{drive_base}spacetimeformer/", exist_ok=True)
    torch.save(stf_model.state_dict(), f"{drive_base}spacetimeformer/stf_model.pt")
    with open(f"{drive_base}spacetimeformer/history.pkl", "wb") as f:
        pickle.dump(stf_history, f)
    mode_str = f"EXOG ({len(stf_exog_cols)} channels)" if stf_exog_cols else "pure-autoregressive"
    print(f"✅ Spacetimeformer trained on train+val only [{mode_str}], checkpointed on best val_loss")


### Step 8 — Spacetimeformer: true held-out test evaluation

The piece that was completely missing before: this forecasts the real held-out `horizon` days
(never seen during training) and scores against real targets from `df_raw`, mirroring Step 4's
TFT evaluation.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell J2 — Spacetimeformer TRUE test evaluation   (V4)
# ──────────────────────────────────────────────────────────────────────────
if STF_AVAILABLE and stf_model is not None:
    # Build the encoder input from the LAST max_encoder_length days of
    # train+val (i.e. immediately preceding the test window), same
    # boundary TFT and the NF models use.
    stf_full = df.pivot_table(index="Date", columns="group_id", values="target")
    stf_full = stf_full[stf_pair_names].dropna(how="any")

    test_dates = sorted(df[df["split"] == "test"]["Date"].unique())[:horizon]
    if len(test_dates) < horizon:
        print(f"⚠️  Only {len(test_dates)} test dates available (expected {horizon}); "
              f"Spacetimeformer test eval will use what's available.")

    history = stf_full[stf_full.index < test_dates[0]].tail(max_encoder_length)
    if len(history) < max_encoder_length:
        print(f"⚠️  Only {len(history)} encoder days available before test start "
              f"(wanted {max_encoder_length}) -- results may be less reliable.")

    # V4: if training used exogenous channels (stf_exog_cols is not None),
    # the test-window encoder input must be built the same way -- same
    # columns, same order, same date alignment -- or the model sees a
    # different d_x than it was trained on and either crashes or silently
    # misreads which channel is which.
    if stf_exog_cols:
        exog_history = (
            df.drop_duplicates("Date")
            .set_index("Date")[stf_exog_cols]
            .reindex(history.index)
        )
        if exog_history.isna().any().any():
            print("⚠️  Missing exogenous values in the test-window encoder history -- "
                  "check date coverage of calendar/gdelt/macro columns near the test "
                  "boundary before trusting this eval.")
        X_test_arr = np.concatenate([history.values, exog_history.values], axis=1)
    else:
        X_test_arr = history.values

    X_test = torch.tensor(X_test_arr, dtype=torch.float32).unsqueeze(0)

    stf_model.eval()
    with torch.no_grad():
        stf_test_out = stf_model(X_test).squeeze(0).numpy()  # (horizon, n_pairs)

    stf_test_preds = pd.DataFrame(
        stf_test_out[: len(test_dates)], index=test_dates, columns=stf_pair_names
    ).reset_index().melt(id_vars="index", var_name="group_id", value_name="stf_pred")
    stf_test_preds = stf_test_preds.rename(columns={"index": "Date"})

    stf_test_scored = stf_test_preds.merge(
        df_raw[["group_id", "Date", "target"]], on=["group_id", "Date"], how="inner"
    )
    n_expected = len(stf_pair_names) * len(test_dates)
    if len(stf_test_scored) != n_expected:
        print(f"⚠️  Spacetimeformer: matched {len(stf_test_scored)} rows against real "
              f"test targets, expected {n_expected} ({len(stf_pair_names)} pairs x "
              f"{len(test_dates)} test days). Check date alignment.")
    else:
        print(f"✅ Spacetimeformer: all {len(stf_test_scored)} test-window forecasts "
              f"matched to real targets")

    stf_test_scored["abs_err"] = (stf_test_scored["stf_pred"] - stf_test_scored["target"]).abs()
    stf_test_mae = stf_test_scored["abs_err"].mean()
    mode_str = f"EXOG ({len(stf_exog_cols)} channels)" if stf_exog_cols else "pure-autoregressive"
    print(f"   Spacetimeformer [{mode_str}] true held-out test MAE: {stf_test_mae:.5f} "
          f"(n={len(stf_test_scored)})")

    # SAVE predictions + config
    stf_test_scored.to_csv(f"{drive_base}spacetimeformer/test_predictions.csv", index=False)
    with open(f"{drive_base}spacetimeformer/config.json", "w") as f:
        json.dump({
            "n_pairs": len(stf_pair_names), "pair_names": stf_pair_names,
            "max_encoder_length": max_encoder_length, "horizon": horizon,
            "exog_mode": bool(stf_exog_cols),
            "exog_cols": stf_exog_cols,
            # FIX: previously missing -- without these, the saved .pt weights
            # can't be loaded back (Spacetimeformer(**arch) must be rebuilt
            # identically before load_state_dict works).
            "architecture": stf_arch_hparams,
            "training": stf_train_hparams,
        }, f, indent=2)
else:
    stf_test_mae = None


### Step 9 — Evaluate NBEATSx / NHITS / PatchTST on the true test window

`nbeats_preds` / `nhits_preds` / `patchtst_preds` from Step 6 were produced by `.predict()`
immediately after fitting on train+val, so they already forecast exactly the `horizon` days that
follow — i.e. the real test window. Here we match those forecasts (using the q50 / median column)
against the real targets in `df_raw`, the same way Step 8 does for Spacetimeformer.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell L — Shared test-evaluation helper for NeuralForecast models
# ──────────────────────────────────────────────────────────────────────────
def evaluate_on_test(preds_df, model_name, df_raw, quantiles=(0.1, 0.5, 0.9)):
    """
    Score a NeuralForecast predictions frame against the real held-out
    targets in df_raw. Returns (scored_df, mae) using the q50 (median)
    column as the point forecast.
    """
    cols = pick_quantile_cols(preds_df, model_name, quantiles=quantiles)
    q50_col = cols[0.5]

    preds = preds_df[["unique_id", "ds", q50_col]].rename(
        columns={"unique_id": "group_id", "ds": "Date", q50_col: f"{model_name}_pred"}
    )
    scored = preds.merge(
        df_raw[["group_id", "Date", "target"]], on=["group_id", "Date"], how="inner"
    )
    n_expected = len(preds)
    if len(scored) != n_expected:
        print(f"⚠️  {model_name}: matched {len(scored)} / {n_expected} rows against "
              f"real test targets. Check date alignment.")
    else:
        print(f"✅ {model_name}: all {len(scored)} test-window forecasts matched to real targets")

    scored["abs_err"] = (scored[f"{model_name}_pred"] - scored["target"]).abs()
    mae = scored["abs_err"].mean()
    print(f"   {model_name} true held-out test MAE: {mae:.5f} (n={len(scored)})")

    scored.to_csv(f"{drive_base}{model_name.lower()}/test_scored.csv", index=False)
    return scored, mae


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell M — NBEATSx: true held-out test evaluation
# ──────────────────────────────────────────────────────────────────────────
nbeats_scored, nbeats_mae = evaluate_on_test(nbeats_preds, "NBEATSx", df_raw)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell N — NHITS: true held-out test evaluation
# ──────────────────────────────────────────────────────────────────────────
nhits_scored, nhits_mae = evaluate_on_test(nhits_preds, "NHITS", df_raw)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell O — PatchTST: true held-out test evaluation
# ──────────────────────────────────────────────────────────────────────────
patchtst_scored, patchtst_mae = evaluate_on_test(patchtst_preds, "PatchTST", df_raw)


### Step 10 — Ensemble the q50 forecasts (NBEATSx + NHITS + PatchTST)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell K — Ensemble, robust quantile parsing
# ──────────────────────────────────────────────────────────────────────────
def q50_frame(preds_df, model_name):
    cols = pick_quantile_cols(preds_df, model_name, quantiles=(0.5,))
    q50_col = cols[0.5]
    return preds_df[["unique_id", "ds", q50_col]].rename(columns={q50_col: f"{model_name}_q50"})

merged = q50_frame(nbeats_preds, "NBEATSx") \
    .merge(q50_frame(nhits_preds, "NHITS"), on=["unique_id", "ds"], how="inner") \
    .merge(q50_frame(patchtst_preds, "PatchTST"), on=["unique_id", "ds"], how="inner")

merged["ensemble_q50"] = merged[["NBEATSx_q50", "NHITS_q50", "PatchTST_q50"]].mean(axis=1)
merged["model_spread"] = (
    merged[["NBEATSx_q50", "NHITS_q50", "PatchTST_q50"]].max(axis=1)
    - merged[["NBEATSx_q50", "NHITS_q50", "PatchTST_q50"]].min(axis=1)
)

print(f"✅ Ensemble merged on (unique_id, ds): {merged.shape}")
print(merged.head(10))

merged.to_csv(f"{drive_base}ensemble/predictions.csv", index=False)


### Step 11 — Unified leaderboard

TFT, NBEATSx, NHITS, PatchTST, and Spacetimeformer test performance side by side — all computed
from the *same* `horizon` held-out days.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell P — Unified leaderboard, same held-out days for every model
# ──────────────────────────────────────────────────────────────────────────
def _extract_tft_mae(test_metrics):
    """test_metrics is a list of dicts from lightning's trainer.test()."""
    if not test_metrics:
        return None
    m = test_metrics[0]
    for key in m:
        if "mae" in key.lower():
            return m[key]
    return None

# V4: "features" column makes feature parity explicit instead of assumed --
# see the V4 changelog cell at the top of the notebook. TFT/NBEATSx/NHITS/
# PatchTST all get the same three families; Spacetimeformer gets them only
# if STF_USE_EXOG was successfully used (stf_exog_cols is not None).
_stf_features = (
    f"price+calendar+gdelt+macro ({len(stf_exog_cols)} exog ch.)" if stf_exog_cols
    else "price only (autoregressive baseline)"
)
leaderboard_rows = [
    {"model": "TFT",       "test_mae": _extract_tft_mae(test_metrics), "features": "price+calendar+gdelt+macro"},
    {"model": "NBEATSx",   "test_mae": nbeats_mae,   "features": "price+calendar+gdelt+macro"},
    {"model": "NHITS",     "test_mae": nhits_mae,    "features": "price+calendar+gdelt+macro"},
    {"model": "PatchTST",  "test_mae": patchtst_mae, "features": "price+calendar+gdelt+macro"},
    {"model": "Spacetimeformer", "test_mae": stf_test_mae, "features": _stf_features},
]
leaderboard = pd.DataFrame(leaderboard_rows).dropna(subset=["test_mae"])
leaderboard = leaderboard.sort_values("test_mae").reset_index(drop=True)

if leaderboard.loc[leaderboard["model"] == "Spacetimeformer", "features"].eq(
    "price only (autoregressive baseline)"
).any():
    print("ℹ️  Spacetimeformer is on fewer features than the other four models -- "
          "treat it as a control (\"how much does time-series structure alone get "
          "you, with zero exogenous data\"), not a like-for-like comparison, until "
          "STF_USE_EXOG is confirmed working (see Step 7).")

print("\n════════════════════════════════════════════════")
print(" UNIFIED LEADERBOARD -- same held-out test window")
print("════════════════════════════════════════════════")
print(leaderboard.to_string(index=False))

os.makedirs(f"{drive_base}leaderboard/", exist_ok=True)
leaderboard.to_csv(f"{drive_base}leaderboard/leaderboard.csv", index=False)
with open(f"{drive_base}leaderboard.pkl", "wb") as f:
    pickle.dump(leaderboard, f)


### Step 12 — Manifest + consolidated hyperparameters

Writes `manifest.json` (pointing at the actual scored test files, not just raw predictions) and
`hyperparameters.json` (every model's architecture/training config in one place, alongside the
seed used) so this run's leaderboard is fully auditable later.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell Q — Manifest of everything saved this run
# ══════════════════════════════════════════════════════════════════════════
# FIX: previously pointed nbeats/nhits/patchtst "predictions" at the RAW
# quantile forecasts (test_predictions.csv from Cell I), not the file the
# leaderboard MAE actually comes from (test_scored.csv, with real targets
# + abs_err, from Cell L/evaluate_on_test). Also now records each model's
# test MAE directly in the manifest, and writes a single consolidated
# hyperparameters.json so the exact config behind this leaderboard is on
# record even if the notebook itself changes later.
manifest = {
    "run": {
        "seed": SEED,
        "split_config": f"{DATA_DIR}split_config.json",
        # V4: which feature families actually reached each model -- see the
        # V4 changelog cell at the top of the notebook for why this matters.
        "feature_families": {
            "gdelt_cols": gdelt_cols,
            "dcc_garch_leg_cols": dcc_garch_leg_cols,
            "macro_leg_cols": macro_leg_cols,
            "calendar_cols": calendar_cols,
            "regime_cols": regime_cols,
            "global_macro_cols": global_macro_cols,
            "extra_event_cols": extra_event_cols,
            "tft_nf_treatment": (
                "calendar_cols = known-future; gdelt_cols (incl. folded-in "
                "tone_shock_interaction/tone_mentions_interaction) + dcc_garch_leg_cols + "
                "macro_leg_cols + regime_cols + global_macro_cols + extra_event_cols = "
                "historical-only, restricted to each group's own 3 legs where applicable "
                "(global_macro_cols is the one exception -- macro_pressure is a single "
                "global column, not per-leg)"
            ),
            # V6 NOTE: regime_cols/global_macro_cols/extra_event_cols were NOT
            # added to Spacetimeformer's candidate_exog_cols (Cell J) -- STF_USE_EXOG
            # defaults to False pending API confirmation (see Cell J's own caveat),
            # and adding untested columns to an already-uncertain exog path felt
            # like the wrong place to also expand scope. If you turn STF_USE_EXOG on,
            # decide then whether to fold these in too.
            "spacetimeformer_mode": "exog" if stf_exog_cols else "pure_autoregressive_baseline",
            "spacetimeformer_exog_cols": stf_exog_cols,
        },
    },
    "data": {
        "full_dataset": f"{DATA_DIR}full_dataset.parquet",
        "train": f"{DATA_DIR}train.parquet",
        "val": f"{DATA_DIR}val.parquet",
        "test": f"{DATA_DIR}test.parquet",
        "split_config": f"{DATA_DIR}split_config.json",
    },
    "models": {
        "tft": {
            "checkpoint": best_tft_path,
            "dataset_params": f"{drive_base}tft/dataset_params.pkl",
            "hyperparameters": f"{drive_base}tft/hyperparameters.json",
            "test_predictions": f"{drive_base}tft/test_predictions.pkl",
            "test_metrics": f"{drive_base}tft/test_metrics.pkl",
            "test_mae": _extract_tft_mae(test_metrics),
        },
        "nbeats": {
            "model_dir": f"{drive_base}nbeats/",
            "hyperparameters": f"{drive_base}nbeats/hyperparameters.json",
            "raw_predictions": f"{drive_base}nbeats/test_predictions.csv",
            "test_scored": f"{drive_base}nbeats/test_scored.csv",
            "test_mae": nbeats_mae,
        },
        "nhits": {
            "model_dir": f"{drive_base}nhits/",
            "hyperparameters": f"{drive_base}nhits/hyperparameters.json",
            "raw_predictions": f"{drive_base}nhits/test_predictions.csv",
            "test_scored": f"{drive_base}nhits/test_scored.csv",
            "test_mae": nhits_mae,
        },
        "patchtst": {
            "model_dir": f"{drive_base}patchtst/",
            "hyperparameters": f"{drive_base}patchtst/hyperparameters.json",
            "raw_predictions": f"{drive_base}patchtst/test_predictions.csv",
            "test_scored": f"{drive_base}patchtst/test_scored.csv",
            "test_mae": patchtst_mae,
        },
        "spacetimeformer": {
            "weights": f"{drive_base}spacetimeformer/stf_model.pt",
            "config": f"{drive_base}spacetimeformer/config.json",
            "history": f"{drive_base}spacetimeformer/history.pkl",
            "test_scored": f"{drive_base}spacetimeformer/test_predictions.csv",
            "test_mae": stf_test_mae,
        },
    },
    "ensemble": f"{drive_base}ensemble/predictions.csv",
    "leaderboard": f"{drive_base}leaderboard/leaderboard.csv",
    "hyperparameters_consolidated": f"{drive_base}hyperparameters.json",
}
with open(f"{drive_base}manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# ── Consolidated hyperparameters across every model, in one place ─────────
all_hyperparams = {
    "seed": SEED,
    "split_config": {
        "max_encoder_length": max_encoder_length,
        "horizon": horizon,
        "val_window": val_window,
    },
    "tft": tft_hparams,
    "neuralforecast": nf_hparams,
    "spacetimeformer": (
        {"architecture": stf_arch_hparams, "training": stf_train_hparams}
        if STF_AVAILABLE and stf_model is not None else None
    ),
}
with open(f"{drive_base}hyperparameters.json", "w") as f:
    json.dump(all_hyperparams, f, indent=2)

print("✅ Manifest saved. Everything from this run is indexed at:")
print(f"   {drive_base}manifest.json")
print("✅ Consolidated hyperparameters saved at:")
print(f"   {drive_base}hyperparameters.json")
